# Fake News Detector

**Aplicatie de detectare a stirilor false** folosind HuggingFace.

Aplicatia primeste un titlu sau un text scurt **in limba engleza** ca input si returneaza probabilitatea ca textul reprezinta o stire falsa sau reala, folosind un model BERT pre-antrenat.

- **Model:** `hamzab/roberta-fake-news-classification` (BERT/RoBERTa fine-tuned pe dataset ISOT de stiri fake/real)
- **Librarie:** HuggingFace Transformers + PyTorch
- **UI:** ipywidgets (interactiv, in notebook)
- **Nota:** modelul este antrenat in limba engleza - introduceti texte in engleza pentru rezultate corecte.

In [1]:
# Cell 2 - Install & import (ruleaza o singura data, apoi poti comenta linia de install)
# !pip install transformers torch ipywidgets --quiet  # decomenteaza la prima rulare

import torch                                           # PyTorch - motorul de calcul pentru retele neuronale
from transformers import AutoTokenizer, AutoModelForSequenceClassification  # incarca model + tokenizer din HuggingFace
import torch.nn.functional as F                        # functii matematice (softmax) pentru probabilitati
import ipywidgets as widgets                           # widget-uri interactive pentru UI
from IPython.display import display, HTML, clear_output # afisare HTML colorat in notebook

print('Librarii incarcate cu succes!')                # confirmare ca importurile au mers

Librarii incarcate cu succes!


In [2]:
# Cell 3 - Incarcare model BERT pre-antrenat
# Folosim un model RoBERTa (varianta optimizata a BERT) fine-tuned pe dataset ISOT de stiri fake/real
MODEL_NAME = 'hamzab/roberta-fake-news-classification'          # model BERT pre-antrenat de pe HuggingFace Hub

print('Descarc tokenizer...')                                   # mesaj info
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)           # tokenizer-ul transforma text in numere

print('Descarc model (poate dura la prima rulare ~500MB)...')   # primul download, apoi cached local (offline)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)  # modelul BERT pre-antrenat

device = 'cuda' if torch.cuda.is_available() else 'cpu'         # foloseste GPU daca exista, altfel CPU
model.to(device)                                                # mutam modelul pe device-ul ales
model.eval()                                                    # mod evaluare (nu antrenam, doar prezicem)

# === Detectare automata a maparii etichetelor (robust pentru orice model) ===
ID2LABEL = model.config.id2label                                # ex: {0: 'FAKE', 1: 'TRUE'} sau {0: 'LABEL_0', 1: 'LABEL_1'}
FAKE_INDEX, REAL_INDEX = None, None                             # le determinam mai jos
for idx, name in ID2LABEL.items():                              # parcurgem etichetele modelului
    name_low = str(name).lower()                                # convertim la lowercase pentru cautare
    if 'fake' in name_low or 'false' in name_low:               # cuvinte cheie pentru FAKE
        FAKE_INDEX = idx                                        # gasit indexul FAKE
    elif 'real' in name_low or 'true' in name_low:              # cuvinte cheie pentru REAL
        REAL_INDEX = idx                                        # gasit indexul REAL

# Daca modelul are doar etichete generice 'LABEL_0'/'LABEL_1' (fara informatie):
# pentru hamzab/roberta-fake-news-classification: LABEL_0 = FAKE, LABEL_1 = TRUE
if FAKE_INDEX is None or REAL_INDEX is None:                    # daca nu am putut detecta
    FAKE_INDEX = 0                                              # fallback: LABEL_0 = FAKE
    REAL_INDEX = 1                                              # fallback: LABEL_1 = REAL
    print('AVERTISMENT: folosesc mapare implicita (LABEL_0=FAKE, LABEL_1=REAL)')

print(f'Device: {device}')                                      # afisam pe ce ruleaza
print(f'Etichete brute model: {ID2LABEL}')                      # afisam etichetele brute
print(f'Mapare detectata: FAKE=index {FAKE_INDEX}, REAL=index {REAL_INDEX}')  # confirmare
print('Model gata de utilizare! (input asteptat: limba engleza)')  # confirmare + atentionare limba

Descarc tokenizer...


config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Descarc model (poate dura la prima rulare ~500MB)...


pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Device: cpu
Etichete brute model: {0: 'FAKE', 1: 'TRUE'}
Mapare detectata: FAKE=index 0, REAL=index 1
Model gata de utilizare! (input asteptat: limba engleza)


In [3]:
# Cell 4 - Functia de predictie
def predict_news(text):
    """Primeste un text si returneaza (eticheta, procent_incredere, prob_fake, prob_real)."""
    if not text or not text.strip():                                                  # verificam input gol
        raise ValueError('Textul este gol!')                                          # eroare clara
    inputs = tokenizer(text, return_tensors='pt', truncation=True,                    # text -> tensori
                       padding=True, max_length=512)                                  # max 512 token-uri (limita BERT)
    inputs = {k: v.to(device) for k, v in inputs.items()}                             # mutam input-ul pe acelasi device ca modelul
    with torch.inference_mode():                                                      # dezactivam gradientii (mai rapid)
        outputs = model(**inputs)                                                     # rulam textul prin model
    logits = outputs.logits                                                           # scoruri brute de la model
    probs = F.softmax(logits, dim=1)[0]                                               # transformam in probabilitati (suma = 1)
    prob_fake = probs[FAKE_INDEX].item()                                              # probabilitatea FAKE (folosim mapare corecta)
    prob_real = probs[REAL_INDEX].item()                                              # probabilitatea REAL (folosim mapare corecta)
    if prob_fake > prob_real:                                                         # alegem eticheta cu prob mai mare
        label = 'FAKE'                                                                # stire falsa
        confidence = prob_fake * 100                                                  # incredere in procente
    else:
        label = 'REAL'                                                                # stire reala
        confidence = prob_real * 100                                                  # incredere in procente
    return label, confidence, prob_fake * 100, prob_real * 100                        # returnam toate informatiile

# Test rapid pentru a verifica functia
label, conf, pf, pr = predict_news('Scientists discover new planet in solar system')  # exemplu
print(f'Test: {label} ({conf:.1f}% incredere) | FAKE={pf:.1f}% | REAL={pr:.1f}%')      # afisare rezultat

Test: REAL (99.9% incredere) | FAKE=0.1% | REAL=99.9%


In [4]:
# Cell 5 - UI interactiv
text_input = widgets.Textarea(                                                        # zona de text pentru input
    value='', placeholder='Scrie aici titlul sau textul stirii...',                  # text de instructiune
    description='Stire:', layout=widgets.Layout(width='600px', height='100px'))      # marime widget

analyze_button = widgets.Button(                                                      # buton de analiza
    description='Analizeaza', button_style='primary',                                 # stil albastru
    icon='search', layout=widgets.Layout(width='150px'))                              # iconita

output_area = widgets.Output()                                                        # zona unde afisam rezultatul

def on_analyze_click(b):                                                              # functia apelata la click
    with output_area:                                                                 # scriem in zona de output
        clear_output()                                                                # stergem rezultatul anterior
        text = text_input.value.strip()                                               # luam textul din input
        if not text:                                                                  # daca e gol
            print('Te rog introdu un text!')                                          # avertizam utilizatorul
            return                                                                    # iesim din functie
        label, conf, pf, pr = predict_news(text)                                      # rulam predictia
        color = '#d9534f' if label == 'FAKE' else '#5cb85c'                           # rosu daca FAKE, verde daca REAL
        icon = '[FAKE]' if label == 'FAKE' else '[REAL]'                              # eticheta text
        html = f'''
        <div style="border:2px solid {color}; border-radius:8px; padding:15px; font-family:Arial;">
            <h2 style="color:{color}; margin:0;">{icon} {label}</h2>
            <p style="font-size:16px;">Incredere: <b>{conf:.2f}%</b></p>
            <div style="background:#eee; border-radius:5px; overflow:hidden; height:20px; width:100%;">
                <div style="background:{color}; width:{conf}%; height:100%;"></div>
            </div>
            <p style="margin-top:10px; color:#555;">FAKE: {pf:.2f}% &nbsp;|&nbsp; REAL: {pr:.2f}%</p>
        </div>'''                                                                     # HTML cu rezultat colorat + bara
        display(HTML(html))                                                           # afisam HTML in notebook

analyze_button.on_click(on_analyze_click)                                             # legam butonul de functie
display(text_input, analyze_button, output_area)                                      # afisam tot UI-ul

Textarea(value='', description='Stire:', layout=Layout(height='100px', width='600px'), placeholder='Scrie aici…

Button(button_style='primary', description='Analizeaza', icon='search', layout=Layout(width='150px'), style=Bu…

Output()

In [6]:
# Cell 6 - Exemple de test
examples = [                                                                          # lista de titluri de test
    'NASA confirms water found on the surface of Mars',                               # stire reala plauzibila
    'Aliens landed in New York and met with the president yesterday',                 # clar fake
    'Stock markets close higher after Federal Reserve announcement',                  # reala
    'Drinking bleach cures all known diseases, doctors confirm',                      # clar fake
]

for ex in examples:                                                                   # parcurgem fiecare exemplu
    label, conf, pf, pr = predict_news(ex)                                            # facem predictia
    print(f'[{label}] ({conf:.1f}%) -> {ex}')                                         # afisam rezultat scurt

[REAL] (100.0%) -> NASA confirms water found on the surface of Mars
[REAL] (100.0%) -> Aliens landed in New York and met with the president yesterday
[REAL] (100.0%) -> Stock markets close higher after Federal Reserve announcement
[REAL] (100.0%) -> Drinking bleach cures all known diseases, doctors confirm


## Cum functioneaza

1. **Tokenizer** - transforma textul in numere (token-uri) pe care modelul le poate intelege.
2. **BERT** - o retea neuronala pre-antrenata pe miliarde de cuvinte care intelege contextul si sensul limbii engleze. Folosim varianta RoBERTa (Robustly Optimized BERT), o versiune imbunatatita.
3. **Fine-tuning** - modelul `hamzab/roberta-fake-news-classification` a fost antrenat suplimentar pe datasetul ISOT (~45000 articole de stiri reale si false) ca sa invete sa le diferentieze.
4. **Softmax** - transforma scorurile brute ale modelului in probabilitati care insumeaza 100%.
5. **Decizie** - eticheta finala este cea cu probabilitatea mai mare (FAKE sau REAL). Maparea index -> eticheta este detectata automat din `model.config.id2label`.

**Limitari:**
- modelul functioneaza doar in limba engleza
- rezultatele sunt orientative (nu absolute) - nicio retea nu detecteaza fake news 100% corect
- modelul a fost antrenat pe articole din intervalul 2016-2017 (dataset ISOT), deci poate fi mai putin precis pe subiecte foarte recente